In [10]:
import pandas as pd
import numpy as np
import os

np.random.seed(42)
n = 800

# Genotype distribution (Dutta 2024)
genotypes = np.random.choice(['HbSS', 'HbSE', 'HbS-thal', 'HbSA'],
                             size=n, p=[0.87, 0.06, 0.03, 0.04])

# Age distribution (Dutta 2024)
age_group_probs = [0.196, 0.50, 0.212, 0.092]
age_ranges = [(0,10), (10,20), (20,30), (30,70)]
ages = []
for i, prob in enumerate(age_group_probs):
    n_group = int(n * prob)
    ages.extend(np.random.randint(age_ranges[i][0], age_ranges[i][1], n_group).tolist())

while len(ages) < n:
    ages.append(int(np.random.randint(10, 20)))
ages = ages[:n]  # ensure exactly 800
ages = np.array(ages, dtype=object)

# Gender (Dutta 2024)
genders = np.random.choice(['M', 'F'], size=n, p=[0.544, 0.456])

# Districts – Odisha and Assam
odisha_dists = ['Sundargarh', 'Koraput', 'Mayurbhanj', 'Kalahandi', 'Nabarangpur']
assam_dists = ['Dibrugarh', 'Tinsukia', 'Sivasagar', 'Jorhat', 'Golaghat']
all_dists = odisha_dists + assam_dists
dist_probs = [0.12,0.10,0.08,0.07,0.06] + [0.20,0.15,0.10,0.07,0.05]
districts = np.random.choice(all_dists, size=n, p=dist_probs)

# Lab values (from Mishra 2025)
def assign_lab(genotype):
    if genotype == 'HbSS':
        hb = np.random.normal(8.32, 0.21)
        hbs = np.random.normal(86.4, 1.0)
    elif genotype == 'HbSA':
        hb = np.random.normal(11.27, 0.18)
        hbs = np.random.normal(40.31, 0.71)
    else:   # HbSE, HbS-thal – approximate
        hb = np.random.normal(10.0, 1.5)
        hbs = np.random.normal(50, 5)
    return round(hb,1), round(hbs,1)

hb_vals = []
hbs_vals = []
for g in genotypes:
    hb, hbs = assign_lab(g)
    hb_vals.append(hb)
    hbs_vals.append(hbs)

# Symptoms (Dutta 2024)
def symptoms(genotype):
    if genotype == 'HbSS':
        fever = np.random.choice(['Yes','No'], p=[0.612,0.388])
        bone_pain = np.random.choice(['Yes','No'], p=[0.484,0.516])
    else:
        fever = np.random.choice(['Yes','No'], p=[0.2,0.8])
        bone_pain = np.random.choice(['Yes','No'], p=[0.15,0.85])
    return fever, bone_pain

fever_vals = []
bone_pain_vals = []
for g in genotypes:
    f, bp = symptoms(g)
    fever_vals.append(f)
    bone_pain_vals.append(bp)

# Awareness (Mishra 2025)
awareness = []
for _ in range(n):
    r = np.random.random()
    if r < 0.543:
        awareness.append('Never heard')
    else:
        r2 = np.random.random()
        if r2 < (0.0937 / 0.457):
            awareness.append('Knows inherited')
        else:
            awareness.append("Heard but don't know inherited")

# black magic belief - 44.55% among those who heard (mishra)
black_magic = []
for a in awareness:
    if a == 'Never heard':
        black_magic.append('No')
    else:
        black_magic.append(np.random.choice(['Yes','No'], p=[0.4455,0.5545]))

# Screening & treatment
screened = []
for g in genotypes:
    if g == 'HbSS':
        screened.append(np.random.choice(['Yes','No'], p=[0.5,0.5]))
    else:
        # From Bindhani: only 8.89% of carriers had been screened before study
        screened.append(np.random.choice(['Yes','No'], p=[0.0889,0.9111]))

hydroxyurea = []
transfusions = []
for g in genotypes:
    if g == 'HbSS':
        hu = np.random.choice(['Yes','No'], p=[0.5,0.5])
        trans = np.random.choice(['0-4/yr','5-8/yr','9-12/yr','>12/yr'],
                                 p=[0.28,0.40,0.32,0.0])  # from Dutta
    else:
        hu = 'No'
        trans = 'Rarely'
    hydroxyurea.append(hu)
    transfusions.append(trans)

# Lost to follow-up 
ltfu = np.random.choice(['Active','Lost to follow-up'], size=n, p=[0.85,0.15])
#check length
for name, arr in [
    ('genotypes', genotypes), ('ages', ages), ('genders', genders),
    ('districts', districts), ('hb_vals', hb_vals), ('hbs_vals', hbs_vals),
    ('fever_vals', fever_vals), ('bone_pain_vals', bone_pain_vals),
    ('awareness', awareness), ('black_magic', black_magic),
    ('screened', screened), ('hydroxyurea', hydroxyurea),
    ('transfusions', transfusions), ('ltfu', ltfu)
]:
    print(f"{name}: {len(arr)}")
    
df = pd.DataFrame({
    'patient_id': [f'SCD{i:04d}' for i in range(1,n+1)],
    'age': ages,
    'gender': genders,
    'district': districts,
    'genotype': genotypes,
    'hb_g_dl': hb_vals,
    'hbs_percent': hbs_vals,
    'fever': fever_vals,
    'bone_pain': bone_pain_vals,
    'awareness': awareness,
    'belief_black_magic': black_magic,
    'screened': screened,
    'hydroxyurea': hydroxyurea,
    'transfusions_per_year': transfusions,
    'follow_up_status': ltfu
})

# Save to CSV
output_file = 'scd_screening_data.csv'
df.to_csv(output_file, index=False)
print(f"Data saved to {output_file} in {os.getcwd()}")
print(f"Shape: {df.shape}")
print("First 5 rows:")
print(df.head())


Starting synthetic data generation...
Current working directory: C:\Users\HP\aiims gwh
genotypes: 800
ages: 800
genders: 800
districts: 800
hb_vals: 800
hbs_vals: 800
fever_vals: 800
bone_pain_vals: 800
awareness: 800
black_magic: 800
screened: 800
hydroxyurea: 800
transfusions: 800
ltfu: 800
Data saved to scd_screening_data.csv in C:\Users\HP\aiims gwh
Shape: (800, 15)
First 5 rows:
  patient_id age gender    district  genotype  hb_g_dl  hbs_percent fever  \
0    SCD0001   7      F   Dibrugarh      HbSS      8.3         85.5   Yes   
1    SCD0002   9      M    Golaghat  HbS-thal     11.3         55.2   Yes   
2    SCD0003   0      F   Sivasagar      HbSS      7.9         85.5   Yes   
3    SCD0004   9      M   Dibrugarh      HbSS      8.0         85.7   Yes   
4    SCD0005   8      F  Sundargarh      HbSS      8.3         85.0   Yes   

  bone_pain                       awareness belief_black_magic screened  \
0        No                     Never heard                 No       No   


In [12]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, ttest_ind

df = pd.read_csv('scd_screening_data.csv')

# basic check
print("total patients:", len(df))

# genotype
print("\nGenotype wise count:")
print(df['genotype'].value_counts())

# awareness
print("\nAwareness (%):")
aw = df['awareness'].value_counts()
for k,v in aw.items():
    print(f"  {k}: {round(v/len(df)*100,1)}%")

# screening
yes_screened = df[df['screened']=='Yes']
pct = len(yes_screened)/len(df)*100
print(f"\nOverall screened: {round(pct,1)}%")

# district wise
print("\nDistrict wise screening %:")
for dist, grp in df.groupby('district'):
    s = round((grp['screened']=='Yes').sum()/len(grp)*100, 1)
    print(f"  {dist}: {s}%")

# HbSS hydroxyurea
hbss = df[df['genotype']=='HbSS']
hu_pct = round((hbss['hydroxyurea']=='Yes').sum()/len(hbss)*100, 1)
print(f"\nHydroxyurea among HbSS: {hu_pct}%")

# LTFU
ltfu_pct = round((df['follow_up_status']=='Lost to follow-up').sum()/len(df)*100, 1)
print(f"\nLost to follow-up: {ltfu_pct}%")

# ---------- stats ----------

# heard vs never heard
df['heard_scd'] = df['awareness'].apply(lambda x: 0 if x=='Never heard' else 1)

# chi square - gender and awareness
ct1 = pd.crosstab(df['gender'], df['heard_scd'])
c1, p1, d1, e1 = chi2_contingency(ct1)
print(f"\nChi-square (gender vs awareness):")
print(f"  chi2 = {round(c1,3)}, p = {round(p1,4)}, df = {d1}")

# age comparison HbSS vs HbSA
age_ss = df[df['genotype']=='HbSS']['age'].dropna()
age_sa = df[df['genotype']=='HbSA']['age'].dropna()
t1, pt = ttest_ind(age_ss, age_sa)
print(f"\nIndependent t-test (age: HbSS vs HbSA):")
print(f"  mean HbSS = {round(age_ss.mean(),1)}, mean HbSA = {round(age_sa.mean(),1)}")
print(f"  t = {round(t1,3)}, p = {round(pt,4)}")

# chi square - black magic belief and screening
ct2 = pd.crosstab(df['belief_black_magic'], df['screened'])
c2, p2, d2, e2 = chi2_contingency(ct2)
print(f"\nChi-square (black magic belief vs screened):")
print(f"  chi2 = {round(c2,3)}, p = {round(p2,4)}, df = {d2}")

total patients: 800

Genotype wise count:
genotype
HbSS        688
HbSE         53
HbSA         34
HbS-thal     25
Name: count, dtype: int64

Awareness (%):
  Never heard: 55.6%
  Heard but don't know inherited: 33.8%
  Knows inherited: 10.6%

Overall screened: 47.0%

District wise screening %:
  Dibrugarh: 50.3%
  Golaghat: 35.3%
  Jorhat: 51.1%
  Kalahandi: 50.8%
  Koraput: 40.9%
  Mayurbhanj: 40.6%
  Nabarangpur: 42.3%
  Sivasagar: 45.6%
  Sundargarh: 50.5%
  Tinsukia: 51.2%

Hydroxyurea among HbSS: 49.9%

Lost to follow-up: 16.6%

Chi-square (gender vs awareness):
  chi2 = 0.755, p = 0.3849, df = 1

Independent t-test (age: HbSS vs HbSA):
  mean HbSS = 18.0, mean HbSA = 12.9
  t = 2.306, p = 0.0214

Chi-square (black magic belief vs screened):
  chi2 = 0.006, p = 0.9362, df = 1


In [19]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, ttest_ind

with open('analysis_output.txt', 'w') as f:
    def write_line(line):
        print(line)
        f.write(line + '\n')

    df = pd.read_csv('scd_screening_data.csv')

    write_line("total patients: " + str(len(df)))

    write_line("\nGenotype wise count:")
    for k, v in df['genotype'].value_counts().items():
        write_line(f"  {k}: {v}")

    write_line("\nAwareness (%):")
    aw = df['awareness'].value_counts()
    for k, v in aw.items():
        write_line(f"  {k}: {round(v/len(df)*100,1)}%")

    yes_screened = df[df['screened']=='Yes']
    pct = len(yes_screened)/len(df)*100
    write_line(f"\nOverall screened: {round(pct,1)}%")

    write_line("\nDistrict wise screening %:")
    for dist, grp in df.groupby('district'):
        s = round((grp['screened']=='Yes').sum()/len(grp)*100, 1)
        write_line(f"  {dist}: {s}%")

    hbss = df[df['genotype']=='HbSS']
    hu_pct = round((hbss['hydroxyurea']=='Yes').sum()/len(hbss)*100, 1) if len(hbss)>0 else 0
    write_line(f"\nHydroxyurea among HbSS: {hu_pct}%")

    ltfu_pct = round((df['follow_up_status']=='Lost to follow-up').sum()/len(df)*100, 1)
    write_line(f"\nLost to follow-up: {ltfu_pct}%")

    # Stats
    df['heard_scd'] = df['awareness'].apply(lambda x: 0 if x=='Never heard' else 1)

    ct1 = pd.crosstab(df['gender'], df['heard_scd'])
    c1, p1, d1, e1 = chi2_contingency(ct1)
    write_line(f"\nChi-square (gender vs awareness):")
    write_line(f"  chi2 = {round(c1,3)}, p = {round(p1,4)}, df = {d1}")

    age_ss = df[df['genotype']=='HbSS']['age'].dropna()
    age_sa = df[df['genotype']=='HbSA']['age'].dropna()
    t1, pt = ttest_ind(age_ss, age_sa)
    write_line(f"\nIndependent t-test (age: HbSS vs HbSA):")
    write_line(f"  mean HbSS = {round(age_ss.mean(),1)}, mean HbSA = {round(age_sa.mean(),1)}")
    write_line(f"  t = {round(t1,3)}, p = {round(pt,4)}")

    ct2 = pd.crosstab(df['belief_black_magic'], df['screened'])
    c2, p2, d2, e2 = chi2_contingency(ct2)
    write_line(f"\nChi-square (black magic belief vs screened):")
    write_line(f"  chi2 = {round(c2,3)}, p = {round(p2,4)}, df = {d2}")

total patients: 800

Genotype wise count:
  HbSS: 688
  HbSE: 53
  HbSA: 34
  HbS-thal: 25

Awareness (%):
  Never heard: 55.6%
  Heard but don't know inherited: 33.8%
  Knows inherited: 10.6%

Overall screened: 47.0%

District wise screening %:
  Dibrugarh: 50.3%
  Golaghat: 35.3%
  Jorhat: 51.1%
  Kalahandi: 50.8%
  Koraput: 40.9%
  Mayurbhanj: 40.6%
  Nabarangpur: 42.3%
  Sivasagar: 45.6%
  Sundargarh: 50.5%
  Tinsukia: 51.2%

Hydroxyurea among HbSS: 49.9%

Lost to follow-up: 16.6%

Chi-square (gender vs awareness):
  chi2 = 0.755, p = 0.3849, df = 1

Independent t-test (age: HbSS vs HbSA):
  mean HbSS = 18.0, mean HbSA = 12.9
  t = 2.306, p = 0.0214

Chi-square (black magic belief vs screened):
  chi2 = 0.006, p = 0.9362, df = 1
